In [ ]:
import cv2
import os
from pathlib import Path
import shutil
import random

In [ ]:
cwd = Path.cwd()

if cwd.name.lower() == "src":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

print("Project root:", PROJECT_ROOT)

# =========================
# CONFIG
# =========================
VIDEOS_DIR = PROJECT_ROOT / "data" / "raw_videos"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "extracted_frames"

SAVE_EVERY = 10
VIDEO_EXTENSIONS = [".mp4", ".mov", ".avi", ".mkv"]

print("Videos folder:", VIDEOS_DIR)
print("Output folder:", OUTPUT_ROOT)

if not VIDEOS_DIR.exists():
    raise FileNotFoundError(f"Videos folder not found: {VIDEOS_DIR}")

# =========================
# FUNCTION TO EXTRACT FRAMES
# =========================
def extract_frames_from_video(video_path, output_root, save_every=10):
    video_name = video_path.stem
    output_dir = Path(output_root) / video_name
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"Could not open video: {video_path}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"\nProcessing: {video_path.name}")
    print(f"FPS: {fps}")
    print(f"Total frames: {total_frames}")
    print(f"Saving into: {output_dir}")

    frame_id = 0
    saved_id = 0

    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if frame_id % save_every == 0:
            save_path = output_dir / f"frame_{saved_id:05d}.jpg"
            cv2.imwrite(str(save_path), frame)
            saved_id += 1

        frame_id += 1

    cap.release()
    print(f"Saved {saved_id} frames for {video_path.name}")


# =========================
# MAIN LOOP OVER ALL VIDEOS
# =========================
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for video_path in VIDEOS_DIR.iterdir():
    if video_path.is_file() and video_path.suffix.lower() in VIDEO_EXTENSIONS:
        extract_frames_from_video(video_path, OUTPUT_ROOT, SAVE_EVERY)

print("\nAll videos processed.")


Processing: Vid_1.mp4
FPS: 25.0
Total frames: 288
Saving into: extracted_frames\Vid_1
Saved 29 frames for Vid_1.mp4

All videos processed.


In [4]:
import shutil
import random
from pathlib import Path

# =========================
# PROJECT ROOT
# =========================
cwd = Path.cwd()

if cwd.name.lower() == "src":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd

print("Project root:", PROJECT_ROOT)

# =========================
# PATHS
# =========================
extracted_root = PROJECT_ROOT / "data" / "extracted_frames"
dataset_root = PROJECT_ROOT / "data" / "yolo_dataset"

print("Extracted frames root:", extracted_root)
print("Dataset root:", dataset_root)

if not extracted_root.exists():
    raise FileNotFoundError(f"Extracted frames folder not found: {extracted_root}")

# =========================
# CLEAN OLD DATASET FILES
# =========================
for split in ["train", "val", "test"]:
    img_dir = dataset_root / "images" / split
    lbl_dir = dataset_root / "labels" / split

    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for f in img_dir.glob("*"):
        f.unlink()

    for f in lbl_dir.glob("*"):
        f.unlink()

# =========================
# COLLECT LABELED IMAGES
# =========================
image_extensions = [".jpg", ".jpeg", ".png"]
labeled_images = []

for video_folder in extracted_root.iterdir():
    if not video_folder.is_dir():
        continue

    labels_src = video_folder / "labels"

    if not labels_src.exists():
        print(f"Skipping {video_folder.name}: labels folder not found")
        continue

    for img_path in video_folder.iterdir():
        if img_path.suffix.lower() not in image_extensions:
            continue

        label_path = labels_src / f"{img_path.stem}.txt"

        if not label_path.exists():
            print(f"Skipping: no label file for {video_folder.name}/{img_path.name}")
            continue

        if label_path.stat().st_size == 0:
            print(f"Skipping: empty label file for {video_folder.name}/{img_path.name}")
            continue

        if label_path.read_text().strip() == "":
            print(f"Skipping: blank label file for {video_folder.name}/{img_path.name}")
            continue

        labeled_images.append((img_path, label_path))

print(f"\nFound {len(labeled_images)} valid labeled images.")

if len(labeled_images) == 0:
    raise ValueError("No valid labeled images found. Check image and label paths.")

# =========================
# SPLIT DATA
# =========================
random.seed(42)
random.shuffle(labeled_images)

n = len(labeled_images)

train_end = int(0.7 * n)
val_end = int(0.9 * n)

splits = {
    "train": labeled_images[:train_end],
    "val": labeled_images[train_end:val_end],
    "test": labeled_images[val_end:]
}

# =========================
# COPY IMAGES + LABELS
# =========================
for split, items in splits.items():
    print(f"\nProcessing split: {split}")

    for img_path, label_path in items:
        new_name = f"{img_path.parent.name}_{img_path.name}"

        dst_img = dataset_root / "images" / split / new_name
        dst_label = dataset_root / "labels" / split / f"{Path(new_name).stem}.txt"

        shutil.copy2(img_path, dst_img)
        shutil.copy2(label_path, dst_label)

        print(f"Copied: {new_name}")

    print(f"{split}: {len(items)} images")

print("\nDataset prepared successfully.")

Project root: c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection
Extracted frames root: c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\data\extracted_frames
Dataset root: c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\data\yolo_dataset
Skipping: no label file for video1/frame_00019.jpg
Skipping: no label file for video1/frame_00020.jpg
Skipping: no label file for video1/frame_00021.jpg
Skipping: no label file for video1/frame_00022.jpg
Skipping: no label file for video1/frame_00023.jpg
Skipping: no label file for video1/frame_00024.jpg
Skipping: no label file for video1/frame_00025.jpg
Skipping: no label file for video1/frame_00026.jpg
Skipping: no label file for video1/frame_00027.jpg
Skipping: no label file for video1/frame_00028.jpg
Skipping: no label file for video1/frame_00032.jpg
Skipping: no label file for video1/frame_00033.jpg
Skipping: no label file for video1/frame_00035.jpg
Skipping: no label file 

In [5]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

False


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
from ultralytics import YOLO
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "src":
    PROJECT_ROOT = PROJECT_ROOT.parent

model = YOLO(PROJECT_ROOT / "models" / "yolov8n.pt")

model.train(
    data=PROJECT_ROOT / "data" / "yolo_dataset" / "data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    project=PROJECT_ROOT / "runs" / "detect",
    name="bowling_yolov8n",
    device=0   
)

EOFError: Ran out of input

In [26]:
from ultralytics import YOLO

model = YOLO(r"runs/detect/flow_test_cpu-4/weights/best.pt")

model.predict(
    source=r"extracted_frames/Vid_1",
    save=True,
    save_txt=True,
    conf=0.25,
    device="cpu"
)


image 1/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00016.jpg: 384x640 1 Giraffe, 54.9ms
image 2/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00017.jpg: 384x640 1 Giraffe, 46.3ms
image 3/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00018.jpg: 384x640 1 Giraffe, 44.8ms
image 4/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00019.jpg: 384x640 1 Giraffe, 46.6ms
image 5/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00020.jpg: 384x640 1 Giraffe, 41.9ms
image 6/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bowling_Detection\src\extracted_frames\Vid_1\frame_00021.jpg: 384x640 1 Giraffe, 44.0ms
image 7/13 c:\Users\Asus\OneDrive\Desktop\New folder\Mobile_RC-Car_Bo

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Giraffe'}
 obb: None
 orig_img: array([[[221, 195, 158],
         [221, 195, 158],
         [221, 195, 158],
         ...,
         [232, 225, 186],
         [232, 225, 186],
         [232, 225, 186]],
 
        [[221, 195, 158],
         [221, 195, 158],
         [221, 195, 158],
         ...,
         [232, 225, 186],
         [232, 225, 186],
         [232, 225, 186]],
 
        [[221, 195, 158],
         [221, 195, 158],
         [221, 195, 158],
         ...,
         [232, 225, 186],
         [232, 225, 186],
         [232, 225, 186]],
 
        ...,
 
        [[ 78,  99, 130],
         [ 78,  99, 130],
         [ 78,  99, 130],
         ...,
         [ 73, 104, 129],
         [ 74, 104, 129],
         [ 74, 104, 129]],
 
        [[ 78,  99, 130],
         [ 78,  99, 130],
         [ 78,  99, 130],
         ...,
         [ 74, 10